In [ ]:
# para instalar se necessário
# ! pip3 install tensorflow
# ! pip3 install mljar-scikit-plot

In [ ]:
import pandas as pd 
import numpy as np 
import tensorflow as tf
import matplotlib.pyplot as plt 
import seaborn as sns
import scikitplot as skplt 
from tensorflow.keras.layers import Dropout
from sklearn.model_selection import train_test_split 
from sklearn.metrics import classification_report
from sklearn.utils import resample
from sklearn import preprocessing

In [ ]:
dados = pd.read_csv('C:/Users/abalb/Desktop/seguro_veiculos_trein.csv')

In [ ]:
dados.head()

In [ ]:
dados.shape

In [ ]:
dados.describe()

In [ ]:
dados.isnull().any()

In [ ]:
dados.Response.value_counts()

In [ ]:
dados.Response.value_counts().plot.pie(explode=[0,0.1], autopct="%1.1f%%", shadow=True)

In [ ]:
sns.countplot(x=dados['Response'])

In [ ]:
dados.drop(['id'],axis=1,inplace=True)

In [ ]:
label_encoder = preprocessing.LabelEncoder()
dados['Gender']=label_encoder.fit_transform(dados['Gender'])
dados['Vehicle_Age']=label_encoder.fit_transform(dados['Vehicle_Age'])
dados['Vehicle_Damage']=label_encoder.fit_transform(dados['Vehicle_Damage'])

In [ ]:
def RNA():
    rna = tf.keras.models.Sequential()
    rna.add(tf.keras.layers.Dense(units=100, activation='relu'))
    rna.add(tf.keras.layers.Dropout(0.3))
    rna.add(tf.keras.layers.Dense(units=75, activation='relu'))
    rna.add(tf.keras.layers.Dropout(0.2))
    rna.add(tf.keras.layers.Dense(units=50, activation='relu'))
    rna.add(tf.keras.layers.Dropout(0.3))
    rna.add(tf.keras.layers.Dense(units=25, activation='relu'))
    rna.add(tf.keras.layers.Dropout(0.2))
    rna.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))
    rna.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return rna

In [ ]:
def metrica(pred_atual, pred_modelo):
    skplt.metrics.plot_confusion_matrix(pred_atual, pred_modelo, title='Matriz de Confusão')
    print(classification_report(pred_atual, pred_modelo))

In [ ]:
rna = RNA()

In [ ]:
vars_interesse = dados.iloc[:,0:10]
alvo = dados.iloc[:,10]

In [ ]:
x_trein, x_teste, y_trein, y_teste = train_test_split(vars_interesse, alvo, test_size=0.2, random_state=100)

In [ ]:
y_trein.value_counts()

In [ ]:
y_teste.value_counts()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
normalizacao = MinMaxScaler()
x_trein = normalizacao.fit_transform(x_trein)
x_teste = normalizacao.fit_transform(x_teste)

In [ ]:
rna.fit((np.array(x_trein)), (np.array(y_trein)), batch_size=32, epochs = 5)

In [ ]:
pred = rna.predict(x_teste)
pred_final = np.round(pred)

In [ ]:
metrica(y_teste, pred_final)

In [ ]:
vars_interesse = dados.iloc[:,0:10]
alvo = dados.iloc[:,10]
x_trein, x_teste, y_trein, y_teste = train_test_split(vars_interesse, alvo, test_size=0.2, random_state=100)

In [ ]:
dados_trein = pd.concat([x_trein, y_trein], axis=1)
dados_majoritarios = dados_trein[dados_trein.iloc[:,10]==0]
dados_minoritarios = dados_trein[dados_trein.iloc[:,10]==1]

In [ ]:
dados_majoritarios_subamostrados = resample(dados_majoritarios, n_samples = len(dados_minoritarios))

In [ ]:
x_trein_subamostrado = pd.concat([dados_majoritarios_subamostrados, dados_minoritarios])

In [ ]:
x_trein_subamostrado['Response'].value_counts()

In [ ]:
x_trein = x_trein_subamostrado.iloc[:,0:10]
y_trein = x_trein_subamostrado.iloc[:,10]

In [ ]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(sampling_strategy = 'minority', k_neighbors = 10)
x_trein, y_trein = smote.fit_resample(x_trein, y_trein)

In [ ]:
from imblearn.over_sampling import ADASYN
adasyn = ADASYN(random_state = 42)
x_trein, y_trein = adasyn.fit_resample(x_trein, y_trein)